In [22]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [19]:
# data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# model
model = xgb.XGBClassifier(
    n_estimators=100,      # number of trees
    learning_rate=0.1,     # step size
    max_depth=6,           # tree depth
    random_state=42
)

model.fit(X_train, y_train)

# predict 
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
accuracy = accuracy_score(y_test, y_pred)

In [25]:
target_names = data.target_names

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


MODEL EVALUATION
Training Accuracy: 100.00%
Test Accuracy: 94.74%
Overfitting Gap: 0.0526
🔴 SIGNIFICANT OVERFITTING (5-10%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

   malignant       0.95      0.90      0.93        42
      benign       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114


Confusion Matrix:
[[38  4]
 [ 2 70]]
Correct predictions: 108
Total predictions: 114
Accuracy: 0.9474
Accuracy: 94.74%

malignant Accuracy: 0.9048 (90.48%)
benign Accuracy: 0.9722 (97.22%)


| Parameter | What it does | Default | Try Values |
|-----------|--------------|---------|------------|
| `n_estimators` | Number of trees | 100 | 50, 100, 200, 500 |
| `learning_rate` | Step size per tree | 0.3 | 0.01, 0.05, 0.1, 0.3 |
| `max_depth` | Tree depth | 6 | 3, 6, 9, 12 |
| `subsample` | Row sampling fraction | 1.0 | 0.6, 0.8, 1.0 |
| `colsample_bytree` | Feature sampling fraction | 1.0 | 0.6, 0.8, 1.0 |
| `reg_alpha` | L1 regularization | 0.0 | 0.0, 0.1, 1.0, 10.0 |
| `reg_lambda` | L2 regularization | 1.0 | 0.0, 0.1, 1.0, 10.0 |
